In [1]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt 

In [ ]:
'model', 'adm_1', 'rank'

In [ ]:
df_mean = pd.read_csv('predictions/mean_rk.csv.gz')

In [ ]:
df_df = pd.read_csv('predictions/default_rk.csv.gz')

df_df_norm = pd.read_csv('predictions/default_norm_rk.csv.gz')

df_mg = pd.read_csv('predictions/mg_base_rk.csv.gz')

df_ma = pd.read_csv('predictions/ma_base_rk.csv.gz')

df_ma

,adm_1,model,arithmetic_mean_ratio,rank
0,11,3rd_imdc_emap_epidematicos_sarimax_state,0.803841,1.0
1,11,3rd_imdc_pucrio_arbocaster,0.834013,3.0
2,11,3rd_imdc_lncc_lncc_arp26_dengue,0.818201,2.0
3,11,3rd_imdc_ifgw_inframind-proteus,0.964784,6.0
4,11,3rd_imdc_purdue_neuralearth,0.919903,4.0
...,...,...,...,...
548,53,3rd_imdc_emap_xgbsillas,1.048428,12.0
549,53,3rd_imdc_emap_epidematicos_prophet,1.271216,18.0
550,53,3rd_imdc_fgv_sakhal,1.361401,19.0
551,53,3rd_imdc_ceri_returnoftheforecast,1.262719,17.0


In [8]:
dfs = {
    "default": df_df,
    "default_norm": df_df_norm,
    "mg": df_mg,
    "ma": df_ma,
}

comparison = None

for name, df in dfs.items():
    aux = df.rename(columns={"rank": f"rank_{name}"})

    if comparison is None:
        comparison = aux
    else:
        comparison = comparison.merge(
            aux,
            on=["adm_1", "model"],
            how="outer"
        )

comparison.head()

,model,adm_1,validation_test,WIS,rank_default,wis_norm,rank_default_norm,geometric_mean_ratio,rank_mg,arithmetic_mean_ratio,rank_ma
0,3rd_imdc_bsc_ghr,11,all,60.581303,13.0,0.625864,12,1.164670,12.0,1.266139,12.0
1,3rd_imdc_ceri_returnoftheforecast,11,all,63.703752,16.0,1.319291,19,1.445585,15.0,1.824717,16.0
2,3rd_imdc_cornell_bentolab,11,all,41.560161,5.0,0.547065,9,0.959128,7.0,0.969922,7.0
3,3rd_imdc_emap_epidematicos_prophet,11,all,45.198347,8.0,0.465263,5,0.915728,6.0,0.926811,5.0
4,3rd_imdc_emap_epidematicos_sarimax_state,11,all,37.760573,3.0,0.407169,1,0.769598,1.0,0.803841,1.0


In [12]:
comparison.loc[comparison.adm_1 == 11][['adm_1', 'model', 'rank_default',"rank_default_norm", "rank_ma", 'rank_mg']] 

,adm_1,model,rank_default,rank_default_norm,rank_ma,rank_mg
0,11,3rd_imdc_bsc_ghr,13.0,12,12.0,12.0
1,11,3rd_imdc_ceri_returnoftheforecast,16.0,19,16.0,15.0
2,11,3rd_imdc_cornell_bentolab,5.0,9,7.0,7.0
3,11,3rd_imdc_emap_epidematicos_prophet,8.0,5,5.0,6.0
4,11,3rd_imdc_emap_epidematicos_sarimax_state,3.0,1,1.0,1.0
5,11,3rd_imdc_emap_lstm,15.0,14,14.0,13.0
6,11,3rd_imdc_emap_xgbsillas,11.0,13,13.0,14.0
7,11,3rd_imdc_fgv_pattern-blue,19.0,18,18.0,19.0
8,11,3rd_imdc_fgv_sakhal,21.0,21,21.0,21.0
9,11,3rd_imdc_fiocruz_mard,14.0,15,15.0,16.0


In [14]:
comparison.loc[comparison.rank_mg == 1][['adm_1', 'model', 'rank_default',"rank_default_norm", "rank_ma", 'rank_mg']] 

,adm_1,model,rank_default,rank_default_norm,rank_ma,rank_mg
4,11,3rd_imdc_emap_epidematicos_sarimax_state,3.0,1,1.0,1.0
39,12,3rd_imdc_purdue_neuralearth,1.0,3,1.0,1.0
59,13,3rd_imdc_pucrio_arbocaster,1.0,1,1.0,1.0
67,14,3rd_imdc_emap_epidematicos_sarimax_state,1.0,1,1.0,1.0
101,15,3rd_imdc_pucrio_arbocaster,1.0,1,4.0,1.0
105,16,3rd_imdc_bsc_ghr,2.0,1,1.0,1.0
139,17,3rd_imdc_lncc_lncc_arp26_dengue,1.0,1,1.0,1.0
163,21,3rd_imdc_procc_bb_model,1.0,1,1.0,1.0
170,22,3rd_imdc_cornell_bentolab,1.0,1,1.0,1.0
193,23,3rd_imdc_emap_epidematicos_sarimax_state,1.0,1,1.0,1.0


In [17]:
comparison.loc[
    comparison[["rank_mg", "rank_default_norm", "rank_ma"]]
    .eq(1)
    .all(axis=1)
][['adm_1', 'model', 'rank_default',"rank_default_norm", "rank_ma", 'rank_mg']].shape

(15, 6)

In [18]:
comparison.loc[
    comparison[["rank_default", "rank_mg", "rank_default_norm", "rank_ma"]]
    .eq(1)
    .all(axis=1)
][['adm_1', 'model', 'rank_default',"rank_default_norm", "rank_ma", 'rank_mg']].shape

(7, 6)

Footrule Distance

In [21]:
from itertools import combinations


In [22]:
rank_cols = [
    "rank_default",
    "rank_default_norm",
    "rank_mg",
    "rank_ma",
]

results = []

for state, g in comparison.groupby("adm_1"):

    # Matriz de distâncias
    dist = pd.DataFrame(
        0.0,
        index=rank_cols,
        columns=rank_cols,
    )

    for c1, c2 in combinations(rank_cols, 2):
        d = (g[c1] - g[c2]).abs().sum()

        dist.loc[c1, c2] = d
        dist.loc[c2, c1] = d

    # Soma das distâncias para as demais metodologias
    total_distance = dist.sum(axis=1)

    results.append(
        pd.DataFrame({
            "adm_1": state,
            "method": total_distance.index.str.replace("rank_", ""),
            "total_distance": total_distance.values,
        })
    )

footrule_scores = pd.concat(results, ignore_index=True)

footrule_scores.head()

,adm_1,method,total_distance
0,11,default,130.0
1,11,default_norm,102.0
2,11,mg,78.0
3,11,ma,74.0
4,12,default,116.0


Qual metodologia mais divergiu em cada estado?

In [24]:
most_divergent = (
    footrule_scores.loc[
        footrule_scores.groupby("adm_1")["total_distance"].idxmax()
    ]
    .sort_values("adm_1")
)

most_divergent['method'].value_counts()

method
default         14
default_norm     8
mg               4
ma               1
Name: count, dtype: int64